# Trénování modelu pro rozpoznávání gest ruky

**Projekt:** Gesture Remote Controller – ovládání počítače gesty ruky  

---

## Přehled

Tento notebook zpracovává data nasbíraná vlastní webkamerou (souřadnice 21 kloubů ruky)
a trénuje klasifikační model, který rozpozná 5 gest:

| Gesto | Akce | Použití |
|---|---|---|
| posun nahoru | ↑ / scroll nahoru | Hlasitost nahoru, předchozí snímek |
| posun dolu | ↓ / scroll dolů | Hlasitost dolů, další snímek |
| posun doprava | Ctrl+Tab | Přepnutí na pravý tab prohlížeče |
| posun doleva | Ctrl+Shift+Tab | Přepnutí na levý tab prohlížeče |
| pauza | Mezerník | Spuštění / pozastavení videa |

### Původ dat

Data byla **sesbírána vlastní rukou** pomocí skriptu `ml/collect.py`:
1. Webkamera snímá pohyby ruky v reálném čase.
2. MediaPipe HandLandmarker detekuje 21 kloubů ruky v každém snímku.
3. Souřadnice (x, y, z) kloubů se zapisují do CSV spolu s labelem gesta.
4. Celkem bylo nasbíráno **4 999 snímků** (≈ 580–1 140 na každé gesto).

### ML pipeline

1. Načtení dat z CSV
2. Průzkumná analýza dat (EDA)
3. Čištění dat (NaN)
4. Kódování labelů (LabelEncoder)
5. Rozdělení 80 / 20 se stratifikací
6. Škálování (StandardScaler)
7. Trénování (Random Forest)
8. Srovnání s jinými algoritmy
9. Vyhodnocení (accuracy, confusion matrix, classification report)
10. Uložení modelu

## Nastavení prostředí

Tuto buňku spusť jako **první**. V Google Colab nainstaluje potřebné knihovny
a zobrazí dialog pro nahrání souboru `dataset.csv`.  
Při lokálním spuštění tuto buňku přeskoč (knihovny jsou v `requirements.txt`).

In [ ]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'scikit-learn', 'pandas', 'matplotlib', 'seaborn', 'joblib'])
    print('Knihovny nainstalovány.')
    from google.colab import files
    print('Nahraj soubor dataset.csv (je v adresáři data/ projektu):')
    uploaded = files.upload()
    DATASET_PATH = 'dataset.csv'
else:
    # Spuštění z adresáře notebook/ (notebook/ -> data/dataset.csv)
    DATASET_PATH = '../data/dataset.csv'
    if not os.path.isfile(DATASET_PATH):
        DATASET_PATH = 'data/dataset.csv'  # záloha pro spuštění z kořene projektu
    print(f'Cesta k datasetu: {os.path.abspath(DATASET_PATH)}')

## Krok 1 – Načtení dat

Data jsou uložena v souboru `dataset.csv` s 64 sloupci:
- `label` – textový název gesta
- `x0, y0, z0` … `x20, y20, z20` – souřadnice 21 kloubů ruky (normalizované 0–1)

Každý řádek odpovídá jednomu snímku z webkamery.

In [ ]:
import pandas as pd

# Načteme CSV soubor do DataFrame.
# Pandas automaticky rozpozná záhlaví (první řádek = názvy sloupců).
data = pd.read_csv(DATASET_PATH)

print('Počet řádků (snímků):', data.shape[0])
print('Počet sloupců:       ', data.shape[1])
print()
print('Typy dat:')
print(data.dtypes.value_counts())

# Ukázka prvních 5 řádků pro vizuální kontrolu
data.head()

## Krok 2 – Průzkumná analýza dat (EDA)

Před trénováním si musíme odpovědět na klíčové otázky:
1. **Je dataset vyvážený?** – mají všechna gesta přibližně stejný počet snímků?
2. **Jaký je rozsah hodnot?** – x, y jsou 0–1; z může být záporné
3. **Jsou v datech chybějící hodnoty?**

Nevyvážený dataset způsobí, že model bude preferovat třídy s více daty.
Vizualizace nám to rychle ukáže.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Počty snímků na třídu
pocty = data['label'].value_counts().sort_index()
print('Počty snímků na třídu:')
print(pocty.to_string())
print(f'\nCelkem: {pocty.sum()} snímků, {len(pocty)} tříd')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Sloupcový graf počtů
pocty.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Počet snímků na gesto')
axes[0].set_xlabel('Gesto')
axes[0].set_ylabel('Počet snímků')
axes[0].tick_params(axis='x', rotation=20)

# Koláčový graf podílů
axes[1].pie(pocty.values, labels=pocty.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Procentuální zastoupení gest')

plt.tight_layout()
plt.show()

# Základní statistiky numerických příznaků
print('\nStatistiky prvních 6 numerických sloupců:')
data.iloc[:, 1:7].describe().round(4)

In [ ]:
# Housličkový graf (violin plot) – distribuce souřadnic zápěstí napříč gesty
# Housličkový graf zobrazuje jak medián, tak celý tvar distribuce

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, koord, title in zip(axes,
                             ['x0', 'y0', 'z0'],
                             ['Souřadnice X – zápěstí', 'Souřadnice Y – zápěstí', 'Souřadnice Z – zápěstí']):
    sns.violinplot(data=data, x='label', y=koord, ax=ax, palette='Set2')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=15)
    ax.set_xlabel('')

plt.suptitle('Distribuce souřadnic zápěstí pro jednotlivá gesta', y=1.02)
plt.tight_layout()
plt.show()

## Krok 3 – Čištění dat

Reálná data mohou obsahovat **chybějící hodnoty (NaN)** – to nastane tehdy,
když MediaPipe v daném snímku ruku nenašel, ale řádek se přesto zapsal prázdný.

**Proč je NaN problém?** Matematické operace s prázdnou hodnotou nejdou provést.
Většina ML algoritmů by při výskytu NaN okamžitě spadla s chybou.

**Řešení:** Odstraníme všechny řádky, kde se NaN nachází.
Protože máme tisíce snímků na každé gesto, ztráta několika neúplných řádků
výsledek nijak neovlivní.

In [ ]:
# Zjistíme počet chybějících hodnot
pocet_nan = data.isnull().sum().sum()
print('Počet chybějících hodnot před čištěním:', pocet_nan)

# Odstraníme řádky obsahující NaN kdekoliv.
# inplace=True = upravenou tabulku uložíme přímo zpět do proměnné data.
data.dropna(inplace=True)

# Obnovíme indexování řádků od 0 (po smazání by mohl být s "dírami").
# drop=True = starý index zahodíme, nepřidáme ho jako nový sloupec.
data.reset_index(drop=True, inplace=True)

print('Počet řádků po čištění:', data.shape[0])

## Krok 4 – Kódování textových labelů na čísla

Sloupec `label` obsahuje textové řetězce. Počítače pracují pouze s čísly.

**LabelEncoder** ze scikit-learn přiřadí každému textu číslo (abecedně):  
`"posun doleva"` → 0, `"posun doprava"` → 1, `"posun dolu"` → 2, `"posun nahoru"` → 3, `"pauza"` → 4

**Proč ukládáme encoder?** Encoder si pamatuje mapování. Při predikci v aplikaci
ho potřebujeme pro převod čísla zpět na textový název gesta.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Vytvoříme prázdný enkodér
le = LabelEncoder()

# fit_transform: naučí se mapování a rovnou zakóduje hodnoty
data['label'] = le.fit_transform(data['label'])

print('Mapování gest na čísla:')
for cislo, text in enumerate(le.classes_):
    print(f'  {cislo} → {text}')

## Krok 5 – Rozdělení dat na trénovací a testovací sadu (80 / 20)

- **X (features):** 63 numerických sloupců – souřadnice kloubů ruky
- **y (target):** sloupec `label` – číslo gesta, které chceme předpovědět

Kdybychom model trénovali a testovali na **stejných** datech, model by si data
zapamatoval a zdánlivě měl 100% přesnost, ale na nových datech by selhal (**overfitting**).

`stratify=y` zajistí, že obě sady mají stejné procentuální zastoupení gest.

In [ ]:
from sklearn.model_selection import train_test_split

# X = všechny sloupce kromě label (63 numerických příznaků)
X = data.drop('label', axis=1).values

# y = cílová proměnná (číslo gesta)
y = data['label'].values

# Rozdělení 80/20 se stratifikací a pevným semínkem pro reprodukovatelnost
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Trénovací sada (X):', X_train.shape)
print('Testovací sada (X):', X_test.shape)

## Krok 6 – Škálování dat (StandardScaler)

**Proč škálovat?**  
Souřadnice x, y jsou v rozsahu 0–1, ale z (hloubka) může nabývat záporných
hodnot blízkých nule (řádově -0.05). Algoritmy citlivé na škálu hodnot
(SVM, KNN, neuronové sítě) by z tohoto rozdílu trpěly.

**StandardScaler** přetransformuje každý sloupec: průměr = 0, odchylka = 1.

**Kritické pravidlo:** `fit_transform()` voláme **pouze na trénovacích datech**.
Na testovacích voláme pouze `transform()`, aby nedošlo k úniku informací
(data leakage).

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit_transform na trénovacích datech: scaler se naučí průměr a odchylku
X_train = scaler.fit_transform(X_train)

# transform na testovacích datech: stejné parametry, žádné nové učení
X_test = scaler.transform(X_test)

print('Škálování dokončeno.')
print(f'Průměr 1. příznaku po škálování: {X_train[:, 0].mean():.6f}')
print(f'Odchylka 1. příznaku po škálování: {X_train[:, 0].std():.6f}')

## Krok 7 – Trénování modelu: Random Forest Classifier

**Jak Random Forest funguje?**  
Skládá se z mnoha rozhodovacích stromů (zde 100). Každý strom se učí na trochu
jiné podmnožině dat. Při predikci každý strom hlasuje pro jednu třídu
a vyhrává třída s nejvíce hlasy.

**Proč Random Forest pro tento projekt?**
- Vynikající přesnost na tabulkových datech
- Odolný vůči overfittingu díky průměrování stromů
- `predict_proba()` vrací pravděpodobnosti – aplikace ho používá pro práh jistoty
- Rychlé predikce v reálném čase (potřeba pro plynné rozpoznávání)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# n_estimators=100 → 100 rozhodovacích stromů
# random_state=42  → reprodukovatelné výsledky
# n_jobs=-1        → využije všechna CPU jádra
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

print('Trénuji model ...')
model.fit(X_train, y_train)
print('Trénování dokončeno!')

## Krok 8 – Srovnání s jinými algoritmy

Abychom zdůvodnili výběr Random Forestu, porovnáme ho s dalšími algoritmy
na **stejných** přeškálovaných datech.

In [ ]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

kandidati = [
    ('Random Forest (100 stromů)', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('SVM (RBF jádro)',            SVC(kernel='rbf', random_state=42, probability=True)),
    ('K-nejbližší sousedé (k=5)', KNeighborsClassifier(n_neighbors=5)),
    ('Rozhodovací strom',          DecisionTreeClassifier(random_state=42)),
]

vysledky = []
print('Trénuji a hodnotím kandidáty ...')
for nazev, m in kandidati:
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    vysledky.append((nazev, acc))
    print(f'  {nazev:<38}  accuracy = {acc:.4f}')

# Sloupcový graf
nazvy, accuracies = zip(*vysledky)
barvy = ['steelblue' if 'Random' in n else 'lightsteelblue' for n in nazvy]

plt.figure(figsize=(9, 4))
bars = plt.bar(nazvy, accuracies, color=barvy, edgecolor='black')
plt.ylim(min(accuracies) - 0.05, 1.01)
plt.ylabel('Přesnost (accuracy)')
plt.title('Srovnání klasifikačních algoritmů')
plt.xticks(rotation=15, ha='right')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
             f'{acc:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## Krok 9 – Vyhodnocení Random Forest modelu

### Metriky
- **Accuracy** – procento správně klasifikovaných snímků
- **Precision** – z předpovědí "gesto X", kolik bylo správně?
- **Recall** – ze skutečných "gesto X", kolik jsme správně zachytili?
- **F1-score** – harmonický průměr precision a recall

### Matice záměn
Vizualizuje, kde model chybuje – která gesta si plete s jinými.
Ideálně by měla svítit pouze diagonála.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred   = model.predict(X_test)
presnost = accuracy_score(y_test, y_pred)

print(f'Přesnost na testovacích datech: {presnost:.4f}  ({presnost * 100:.1f} %)')
print()
print('Klasifikační zpráva:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Matice záměn ---
matice = confusion_matrix(y_test, y_pred)
sns.heatmap(
    matice,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    ax=axes[0]
)
axes[0].set_xlabel('Předpovězené gesto')
axes[0].set_ylabel('Skutečné gesto')
axes[0].set_title('Matice záměn')
axes[0].tick_params(axis='x', rotation=15)

# --- Feature importance (top 15 příznaků) ---
importances = model.feature_importances_
top_idx = importances.argsort()[-15:][::-1]

# Sestavíme názvy sloupců: x0, y0, z0, x1, y1, z1, ..., x20, y20, z20
sloupce = []
for i in range(21):
    for osa in ['x', 'y', 'z']:
        sloupce.append(f'{osa}{i}')

axes[1].bar(range(15), importances[top_idx], color='steelblue', edgecolor='black')
axes[1].set_xticks(range(15))
axes[1].set_xticklabels([sloupce[i] for i in top_idx], rotation=45, ha='right')
axes[1].set_ylabel('Důležitost příznaku')
axes[1].set_title('Top 15 nejdůležitějších příznaků')

plt.tight_layout()
plt.show()

## Krok 10 – Uložení modelu

Ukládáme tři soubory:

| Soubor | Obsah | Proč je potřeba |
|---|---|---|
| `model.pkl` | Natrénovaný RandomForest | Provádí predikci gesta |
| `scaler.pkl` | Natrénovaný StandardScaler | Vstup musí být škálován **stejným** způsobem jako při trénování |
| `label_encoder.pkl` | LabelEncoder | Převod čísla → textový název gesta |

**Proč `joblib`?** Je optimalizovaný pro velká numpy pole (jako jsou parametry
RandomForestu) – rychlejší a kompaktnější než standardní `pickle`.

Po stažení přesuň soubory do složky `models/` projektu a spusť aplikaci:  
`python run.py`

In [ ]:
import joblib

if IN_COLAB:
    SAVE_DIR = '.'
else:
    SAVE_DIR = '../models'
    os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(model,  os.path.join(SAVE_DIR, 'model.pkl'))
joblib.dump(scaler, os.path.join(SAVE_DIR, 'scaler.pkl'))
joblib.dump(le,     os.path.join(SAVE_DIR, 'label_encoder.pkl'))

print('Uloženo:')
print(f'  {os.path.join(SAVE_DIR, "model.pkl")}')
print(f'  {os.path.join(SAVE_DIR, "scaler.pkl")}')
print(f'  {os.path.join(SAVE_DIR, "label_encoder.pkl")}')

if IN_COLAB:
    from google.colab import files
    files.download('model.pkl')
    files.download('scaler.pkl')
    files.download('label_encoder.pkl')
    print('\nSoubory se stahují. Přesuň je do složky models/ projektu.')

print(f'\n=== HOTOVO === Přesnost modelu: {presnost:.4f} ({presnost*100:.1f} %)')